# from pathlib import Path

DATA_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30"
)

DATA_DIR.exists()


In [ ]:
for f in sorted(DATA_DIR.iterdir()):
    print(f.name)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import re
from collections import defaultdict

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

In [ ]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files\n")

for file in csv_files:
    print(file.name)

In [ ]:
schema = []

for file in csv_files:
    df = pd.read_csv(
        file,
        encoding="latin1",
        nrows=5
    )
    
    for col in df.columns:
        schema.append({
            "file": file.name,
            "column": col
        })

schema_df = pd.DataFrame(schema)

schema_df

In [ ]:
for file in csv_files:
    
    df = pd.read_csv(
        file,
        encoding="latin1",
        nrows=5
    )
    
    print("\n" + "=" * 100)
    print(file.name)
    print("=" * 100)
    print("Columns:")
    print(list(df.columns))

In [ ]:
def load_csv(name):
    path = DATA_DIR / name
    
    return pd.read_csv(
        path,
        encoding="latin1",
        low_memory=False
    )

In [ ]:
food = load_csv("food.csv")
food_attribute = load_csv("food_attribute.csv")
food_attribute_type = load_csv("food_attribute_type.csv")
food_category = load_csv("food_category.csv")

food_calorie_conversion_factor = load_csv(
    "food_calorie_conversion_factor.csv"
)

food_component = load_csv("food_component.csv")
food_nutrient = load_csv("food_nutrient.csv")
food_nutrient_conversion_factor = load_csv(
    "food_nutrient_conversion_factor.csv"
)

food_portion = load_csv("food_portion.csv")
food_protein_conversion_factor = load_csv(
    "food_protein_conversion_factor.csv"
)

foundation_food = load_csv("foundation_food.csv")

input_food = load_csv("input_food.csv")

lab_method = load_csv("lab_method.csv")
lab_method_code = load_csv("lab_method_code.csv")
lab_method_nutrient = load_csv("lab_method_nutrient.csv")

market_acquisition = load_csv("market_acquisition.csv")
measure_unit = load_csv("measure_unit.csv")
nutrient = load_csv("nutrient.csv")

sample_food = load_csv("sample_food.csv")
sub_sample_food = load_csv("sub_sample_food.csv")
sub_sample_result = load_csv("sub_sample_result.csv")

acquisition_samples = load_csv("acquisition_samples.csv")
agricultural_samples = load_csv("agricultural_samples.csv")

food_update_log_entry = load_csv(
    "food_update_log_entry.csv"
)

In [ ]:
tables = {
    "food": food,
    "food_attribute": food_attribute,
    "food_attribute_type": food_attribute_type,
    "food_category": food_category,
    "food_calorie_conversion_factor": food_calorie_conversion_factor,
    "food_component": food_component,
    "food_nutrient": food_nutrient,
    "food_nutrient_conversion_factor": food_nutrient_conversion_factor,
    "food_portion": food_portion,
    "food_protein_conversion_factor": food_protein_conversion_factor,
    "foundation_food": foundation_food,
    "input_food": input_food,
    "lab_method": lab_method,
    "lab_method_code": lab_method_code,
    "lab_method_nutrient": lab_method_nutrient,
    "market_acquisition": market_acquisition,
    "measure_unit": measure_unit,
    "nutrient": nutrient,
    "sample_food": sample_food,
    "sub_sample_food": sub_sample_food,
    "sub_sample_result": sub_sample_result,
    "acquisition_samples": acquisition_samples,
    "agricultural_samples": agricultural_samples,
    "food_update_log_entry": food_update_log_entry,
}

for name, df in tables.items():
    print(
        f"{name:45s} "
        f"rows={len(df):>10,} "
        f"columns={len(df.columns):>3}"
    )

In [ ]:
for name, df in tables.items():
    print("\n")
    print("=" * 100)
    print(name)
    print("=" * 100)
    print(list(df.columns))

In [ ]:
def clean_column_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")


for name, df in tables.items():
    df.columns = [
        clean_column_name(c)
        for c in df.columns
    ]

In [ ]:
for name, df in tables.items():
    print(name)
    print(df.columns.tolist())
    print()

In [ ]:
food.head()

In [ ]:
food.info()

In [ ]:
print("Rows:", len(food))
print("Unique FDC IDs:", food["fdc_id"].nunique())
print("Missing FDC IDs:", food["fdc_id"].isna().sum())
print("Duplicate FDC IDs:", food["fdc_id"].duplicated().sum())

In [ ]:
def id_columns(df):
    return [
        c for c in df.columns
        if (
            c.endswith("_id")
            or c == "id"
            or "fdc_id" in c
        )
    ]


for name, df in tables.items():
    print(f"\n{name}")
    print(id_columns(df))

In [ ]:
food_ids = set(
    food["fdc_id"]
    .dropna()
    .astype("int64")
)

for name, df in tables.items():
    
    if "fdc_id" not in df.columns:
        continue
    
    ids = set(
        pd.to_numeric(
            df["fdc_id"],
            errors="coerce"
        )
        .dropna()
        .astype("int64")
    )
    
    missing = ids - food_ids
    
    print(
        f"{name:45s} "
        f"unique FDC IDs={len(ids):>8,} "
        f"not in food={len(missing):>8,}"
    )

In [ ]:
food_nutrient.head()

In [ ]:
nutrient.head()

In [ ]:
print(food_nutrient.columns.tolist())
print(nutrient.columns.tolist())

In [ ]:
food_nutrient["fdc_id"].nunique()
food_nutrient["nutrient_id"].nunique()

In [ ]:
food_nutrient["nutrient_id"].isna().sum()
food_nutrient["fdc_id"].isna().sum()

In [ ]:
nutrient_ids = set(
    nutrient["id"]
    .dropna()
    .astype("int64")
)

food_nutrient_ids = set(
    food_nutrient["nutrient_id"]
    .dropna()
    .astype("int64")
)

missing_nutrients = food_nutrient_ids - nutrient_ids

print(
    "Nutrient IDs not found in nutrient table:",
    len(missing_nutrients)
)

if missing_nutrients:
    print(sorted(missing_nutrients)[:20])

In [ ]:
nutrient_lookup = nutrient.copy()

nutrient_lookup = nutrient_lookup.rename(
    columns={
        c: f"nutrient_{c}"
        for c in nutrient_lookup.columns
        if c != "id"
    }
)

nutrient_lookup = nutrient_lookup.rename(
    columns={
        "nutrient_id": "nutrient_id"
    }
    if "nutrient_id" in nutrient_lookup.columns
    else {}
)

In [ ]:
nutrient_lookup.head()

In [ ]:
nutrient.columns

In [ ]:
food_nutrient_with_metadata = food_nutrient.merge(
    nutrient_lookup,
    left_on="nutrient_id",
    right_on="id",
    how="left",
    validate="many_to_one"
)

In [ ]:
food_nutrient.columns.tolist()

In [ ]:
food_nutrient.head(20)

In [ ]:
food_nutrient.groupby(
    ["fdc_id", "nutrient_id"]
).size().value_counts().sort_index()

In [ ]:
nutrient_wide = (
    food_nutrient_with_metadata
    .pivot_table(
        index="fdc_id",
        columns="nutrient_name",
        values="amount",
        aggfunc="mean"
    )
    .reset_index()
)

In [ ]:
print(food_nutrient.columns.tolist())

In [ ]:
print(nutrient.columns.tolist())

In [ ]:
for name, df in tables.items():
    print("\n" + "=" * 120)
    print(name)
    print("=" * 120)
    
    print("SHAPE:", df.shape)
    print("COLUMNS:")
    
    for i, col in enumerate(df.columns, 1):
        print(f"{i:3}. {col}")

In [ ]:
important_tables = [
    "food",
    "foundation_food",
    "food_nutrient",
    "nutrient",
    "food_portion",
    "measure_unit",
    "food_attribute",
    "food_attribute_type",
    "food_component",
    "sample_food",
    "sub_sample_food",
    "sub_sample_result",
    "lab_method",
    "lab_method_code",
    "lab_method_nutrient",
    "acquisition_samples",
    "market_acquisition",
    "agricultural_samples",
    "input_food",
]

for name in important_tables:
    df = tables[name]
    
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print(df.head(3).to_string())


In [ ]:
from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30"
)

print("Folder exists:", DATA_DIR.exists())

for f in sorted(DATA_DIR.glob("*.csv")):
    print(f.name)


In [ ]:
def load_csv(filename):
    return pd.read_csv(
        DATA_DIR / filename,
        encoding="latin1",
        low_memory=False
    )


tables = {}

for file in DATA_DIR.glob("*.csv"):
    tables[file.stem] = load_csv(file.name)

print(f"Loaded {len(tables)} CSV files.")


In [ ]:
for name, df in tables.items():
    print(
        f"{name:45s} "
        f"rows = {len(df):>10,}   "
        f"columns = {len(df.columns):>3}"
    )


In [ ]:
for name, df in tables.items():
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())


In [ ]:
important_tables = [
    "food",
    "foundation_food",
    "food_nutrient",
    "nutrient",
    "food_portion",
    "measure_unit",
    "food_attribute",
    "food_attribute_type",
    "food_component",
    "sample_food",
    "sub_sample_food",
    "sub_sample_result",
    "lab_method",
    "lab_method_code",
    "lab_method_nutrient",
    "acquisition_samples",
    "market_acquisition",
    "agricultural_samples",
    "input_food",
]

for name in important_tables:
    print("\n" + "#" * 100)
    print(name)
    print("#" * 100)
    print(tables[name].head(3).to_string(index=False))


In [ ]:
foundation_ids = set(
    foundation_food["fdc_id"]
    .dropna()
    .astype("int64")
)

food_ids = set(
    food["fdc_id"]
    .dropna()
    .astype("int64")
)

print("Foundation Foods:", len(foundation_ids))
print("Food records:", len(food_ids))

print(
    "Foundation food IDs missing from food:",
    len(foundation_ids - food_ids)
)

print(
    "Foundation food IDs found in food:",
    len(foundation_ids & food_ids)
)


In [ ]:
food["data_type"].value_counts(dropna=False)


In [ ]:
food[
    food["fdc_id"].isin(foundation_ids)
]["data_type"].value_counts(dropna=False)


In [ ]:
foundation_food_nutrients = food_nutrient[
    food_nutrient["fdc_id"].isin(foundation_ids)
].copy()

print(
    "Nutrient records belonging directly to Foundation Foods:",
    len(foundation_food_nutrients)
)

print(
    "Foundation Foods with nutrients:",
    foundation_food_nutrients["fdc_id"].nunique()
)

print(
    "Foundation Foods without nutrients:",
    len(
        foundation_ids
        - set(
            foundation_food_nutrients["fdc_id"]
            .dropna()
            .astype("int64")
        )
    )
)


In [ ]:
nutrient_duplicates = (
    foundation_food_nutrients
    .groupby(["fdc_id", "nutrient_id"])
    .size()
    .reset_index(name="count")
)

nutrient_duplicates[
    nutrient_duplicates["count"] > 1
].sort_values(
    "count",
    ascending=False
).head(20)


In [ ]:
print(
    "Unique food/nutrient combinations:",
    len(nutrient_duplicates)
)

print(
    "Food/nutrient combinations with >1 record:",
    (
        nutrient_duplicates["count"] > 1
    ).sum()
)


In [ ]:
foundation_food_nutrients.merge(
    nutrient,
    left_on="nutrient_id",
    right_on="id",
    how="left",
    validate="many_to_one"
).head(20)


In [ ]:
nutrient_check = foundation_food_nutrients.merge(
    nutrient,
    left_on="nutrient_id",
    right_on="id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

nutrient_check["_merge"].value_counts()


In [ ]:
nutrient_check[
    [
        "fdc_id",
        "nutrient_id",
        "amount",
        "name",
        "unit_name",
        "nutrient_nbr"
    ]
].head(30)


In [ ]:
foundation_portions = food_portion[
    food_portion["fdc_id"].isin(foundation_ids)
].copy()

print(
    "Portion records for Foundation Foods:",
    len(foundation_portions)
)

print(
    "Foundation Foods having portions:",
    foundation_portions["fdc_id"].nunique()
)


In [ ]:
portion_unit_check = foundation_portions.merge(
    measure_unit,
    left_on="measure_unit_id",
    right_on="id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

portion_unit_check["_merge"].value_counts()


In [ ]:
foundation_attributes = food_attribute[
    food_attribute["fdc_id"].isin(foundation_ids)
].copy()

print(
    "Attribute records:",
    len(foundation_attributes)
)

print(
    "Foundation Foods with attributes:",
    foundation_attributes["fdc_id"].nunique()
)


In [ ]:
attribute_check = foundation_attributes.merge(
    food_attribute_type,
    left_on="food_attribute_type_id",
    right_on="id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

attribute_check["_merge"].value_counts()


In [ ]:
category_check = (
    food[
        food["fdc_id"].isin(foundation_ids)
    ]
    .merge(
        food_category,
        left_on="food_category_id",
        right_on="id",
        how="left",
        indicator=True,
        validate="many_to_one"
    )
)

category_check["_merge"].value_counts()


In [ ]:
category_check[
    [
        "fdc_id",
        "description",
        "food_category_id",
        "code",
        "food_category"
        if "food_category" in category_check.columns
        else "description_y"
    ]
].head()


In [ ]:
category_check.columns.tolist()


In [ ]:
sample_result_check = sub_sample_result.merge(
    food_nutrient[
        [
            "id",
            "fdc_id",
            "nutrient_id",
            "amount"
        ]
    ],
    left_on="food_nutrient_id",
    right_on="id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

sample_result_check["_merge"].value_counts()


In [ ]:
sample_result_check.head(20)


In [ ]:
sample_ids = set(
    sample_food["fdc_id"]
    .dropna()
    .astype("int64")
)

sub_sample_parent_ids = set(
    sub_sample_food["fdc_id_of_sample_food"]
    .dropna()
    .astype("int64")
)

print(
    "Sub-sample parent IDs not found in sample_food:",
    len(
        sub_sample_parent_ids - sample_ids
    )
)


In [ ]:
sub_sample_check = sub_sample_food.merge(
    sample_food,
    left_on="fdc_id_of_sample_food",
    right_on="fdc_id",
    how="left",
    indicator=True,
    suffixes=("", "_sample"),
)

sub_sample_check["_merge"].value_counts()


In [ ]:
acquisition_sample_check = acquisition_samples.merge(
    food[
        [
            "fdc_id",
            "data_type",
            "description"
        ]
    ],
    left_on="fdc_id_of_sample_food",
    right_on="fdc_id",
    how="left",
    indicator=True,
    suffixes=("", "_sample"),
)

acquisition_sample_check["_merge"].value_counts()


In [ ]:
acquisition_sample_check.head(20)


In [ ]:
acquisition_food_check = acquisition_samples.merge(
    food[
        [
            "fdc_id",
            "data_type",
            "description"
        ]
    ],
    left_on="fdc_id_of_acquisition_food",
    right_on="fdc_id",
    how="left",
    indicator=True,
    suffixes=("", "_acquisition"),
)

acquisition_food_check["_merge"].value_counts()


In [ ]:
foundation_type_ids = set(
    food.loc[
        food["data_type"] == "foundation_food",
        "fdc_id"
    ].dropna().astype("int64")
)

print("food.csv data_type='foundation_food':", len(foundation_type_ids))
print("foundation_food.csv records:", len(foundation_ids))

print(
    "In food.csv but NOT foundation_food.csv:",
    len(foundation_type_ids - foundation_ids)
)

print(
    "In foundation_food.csv but NOT food.csv:",
    len(foundation_ids - foundation_type_ids)
)


In [ ]:
extra_foundation_type_ids = sorted(
    foundation_type_ids - foundation_ids
)

extra_foundation_type_ids[:50]


In [ ]:
food[
    food["fdc_id"].isin(extra_foundation_type_ids)
].sort_values("fdc_id").head(100)


In [ ]:
duplicate_food_nutrients = (
    foundation_food_nutrients
    .groupby(["fdc_id", "nutrient_id"])
    .size()
    .reset_index(name="count")
)

duplicate_food_nutrients = duplicate_food_nutrients[
    duplicate_food_nutrients["count"] > 1
]

duplicate_food_nutrients


In [ ]:
duplicate_pairs = duplicate_food_nutrients[
    ["fdc_id", "nutrient_id"]
].merge(
    foundation_food_nutrients,
    on=["fdc_id", "nutrient_id"],
    how="left"
)

duplicate_pairs


In [ ]:
duplicate_pairs.merge(
    nutrient[
        ["id", "name", "unit_name", "nutrient_nbr"]
    ],
    left_on="nutrient_id",
    right_on="id",
    how="left",
    suffixes=("", "_nutrient")
)[
    [
        "fdc_id",
        "nutrient_id",
        "name",
        "unit_name",
        "amount",
        "data_points",
        "derivation_id",
        "min",
        "max",
        "median",
        "footnote",
        "min_year_acquired"
    ]
]


In [ ]:
unmatched_nutrients = nutrient_check[
    nutrient_check["_merge"] == "left_only"
].copy()

print(unmatched_nutrients.shape)

unmatched_nutrients[
    [
        "fdc_id",
        "nutrient_id",
        "amount",
        "data_points",
        "derivation_id",
        "min",
        "max",
        "median"
    ]
].head(100)


In [ ]:
print(
    "Unique unmatched nutrient IDs:",
    unmatched_nutrients["nutrient_id"].unique()
)


In [ ]:
nutrient[
    nutrient["id"].isin(
        unmatched_nutrients["nutrient_id"].dropna()
    )
]


In [ ]:
orphan_sample_parent_ids = sorted(
    sub_sample_parent_ids - sample_ids
)

print(orphan_sample_parent_ids)


In [ ]:
sub_sample_food[
    sub_sample_food["fdc_id_of_sample_food"].isin(
        orphan_sample_parent_ids
    )
]


In [ ]:
food[
    food["fdc_id"].isin(
        orphan_sample_parent_ids
    )
]


In [ ]:
unmatched_sample_results = sample_result_check[
    sample_result_check["_merge"] == "left_only"
].copy()

unmatched_sample_results


In [ ]:
unmatched_sample_results[
    [
        "food_nutrient_id",
        "adjusted_amount",
        "lab_method_id",
        "nutrient_name"
    ]
]


In [ ]:
sub_sample_result[
    sub_sample_result["food_nutrient_id"].isin(
        unmatched_sample_results["food_nutrient_id"]
    )
]


In [ ]:
foundation_components = food_component[
    food_component["fdc_id"].isin(foundation_ids)
].copy()

print("Foundation component rows:", len(foundation_components))
print(
    "Foundation foods with components:",
    foundation_components["fdc_id"].nunique()
)

print(
    "Rows with missing fdc_id:",
    food_component["fdc_id"].isna().sum()
)


In [ ]:
foundation_components.head(20)


In [ ]:
foundation_inputs = input_food[
    input_food["fdc_id"].isin(foundation_ids)
].copy()

print("Input-food rows:", len(foundation_inputs))
print(
    "Foundation foods with input foods:",
    foundation_inputs["fdc_id"].nunique()
)


In [ ]:
foundation_inputs.head(30)


In [ ]:
input_food_ids = set(
    foundation_inputs["fdc_of_input_food"]
    .dropna()
    .astype("int64")
)

print(
    "Input food IDs not found in food.csv:",
    len(input_food_ids - food_ids)
)


In [ ]:
category_check[
    [
        "fdc_id",
        "description_x",
        "food_category_id",
        "code",
        "description_y"
    ]
].head(20)


In [ ]:
food_update_log_entry.head(20)


In [ ]:
food_update_log_entry["id"].nunique(), len(food_update_log_entry)


In [ ]:
set(food_update_log_entry["id"]) == set(food["fdc_id"])


In [ ]:
foundation_ids = set(foundation_food["fdc_id"].dropna().astype(int))

food_foundation = food[
    food["data_type"].eq("foundation_food")
].copy()

food_foundation["in_foundation_table"] = (
    food_foundation["fdc_id"].astype(int).isin(foundation_ids)
)

print(
    food_foundation["in_foundation_table"]
    .value_counts()
)

extra_foundation = food_foundation[
    ~food_foundation["in_foundation_table"]
].copy()

print("Extra records:", len(extra_foundation))

display(
    extra_foundation[
        ["fdc_id", "description", "publication_date"]
    ].sort_values("fdc_id")
)


In [ ]:
nutrient[nutrient["id"] == 2066]


In [ ]:
print("Nutrient 2066 in nutrient.csv:")
display(nutrient[nutrient["id"] == 2066])

print("\nFood nutrient records:")
display(
    food_nutrient[
        food_nutrient["nutrient_id"] == 2066
    ]
)


In [ ]:
foundation_ids = set(foundation_food["fdc_id"])

nutrient_2066 = food_nutrient[
    food_nutrient["nutrient_id"] == 2066
].copy()

nutrient_2066["is_foundation"] = (
    nutrient_2066["fdc_id"].isin(foundation_ids)
)

display(nutrient_2066)
print(
    nutrient_2066["is_foundation"].value_counts()
)


In [ ]:
duplicate_pairs = (
    food_nutrient
    .groupby(["fdc_id", "nutrient_id"])
    .size()
    .reset_index(name="count")
)

duplicate_pairs = duplicate_pairs[
    duplicate_pairs["count"] > 1
]

display(duplicate_pairs)


In [ ]:
for _, row in duplicate_pairs.iterrows():

    fdc = int(row["fdc_id"])
    nutrient_id = int(row["nutrient_id"])

    print("\n" + "=" * 80)
    print("FDC ID:", fdc)
    print("Nutrient ID:", nutrient_id)

    display(
        food_nutrient[
            (food_nutrient["fdc_id"] == fdc) &
            (food_nutrient["nutrient_id"] == nutrient_id)
        ]
    )


In [ ]:
missing_parent = sub_sample_food[
    ~sub_sample_food["fdc_id_of_sample_food"]
    .isin(sample_food["fdc_id"])
]

display(missing_parent)

missing_parent_id = (
    missing_parent["fdc_id_of_sample_food"]
    .dropna()
    .unique()
)

print(missing_parent_id)


In [ ]:
display(
    food[
        food["fdc_id"].isin(missing_parent_id)
    ]
)


In [ ]:
display(
    food[
        food["fdc_id"].isin(
            missing_parent["fdc_id"]
        )
    ]
)


In [ ]:
unmatched_results = sub_sample_result[
    ~sub_sample_result["food_nutrient_id"]
    .isin(food_nutrient["id"])
].copy()

print("Unmatched:", len(unmatched_results))

display(unmatched_results)


In [ ]:
print(
    unmatched_results["nutrient_name"].value_counts()
)


In [ ]:
display(
    unmatched_results[
        [
            "food_nutrient_id",
            "adjusted_amount",
            "lab_method_id",
            "nutrient_name"
        ]
    ]
)


In [ ]:
[
    'fdc_id',
    'data_type',
    'description_x',
    'food_category_id',
    'publication_date',
    'id',
    'code',
    'description_y',
    '_merge'
]


In [ ]:
category_check["description"]


In [ ]:
category_check[
    [
        "fdc_id",
        "description_x",
        "food_category_id",
        "code",
        "description_y"
    ]
].head()


In [ ]:
category_check = category_check.rename(
    columns={
        "description_x": "food_description",
        "description_y": "category_description"
    }
)

display(
    category_check[
        [
            "fdc_id",
            "food_description",
            "food_category_id",
            "code",
            "category_description"
        ]
    ].head()
)


In [ ]:
print(food_component["fdc_id"].isna().sum())
print(food_component["fdc_id"].notna().sum())

display(
    food_component[
        food_component["fdc_id"].notna()
    ].head()
)


In [ ]:
display(food_component.head(20))


In [ ]:
print("=" * 70)
print("FINAL FOUNDATION FOOD INTEGRITY SUMMARY")
print("=" * 70)

print("Foundation foods:", len(foundation_food))
print("Foundation IDs in food:",
      foundation_food["fdc_id"].isin(food["fdc_id"]).sum())

print("Foundation foods with nutrients:",
      food_nutrient[
          food_nutrient["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Foundation foods with portions:",
      food_portion[
          food_portion["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Foundation foods with input foods:",
      input_food[
          input_food["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Foundation foods with attributes:",
      food_attribute[
          food_attribute["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Foundation foods with components:",
      food_component[
          food_component["fdc_id"].isin(foundation_ids)
      ]["fdc_id"].nunique())

print("Unmatched nutrient IDs:",
      sorted(
          set(food_nutrient["nutrient_id"].dropna())
          - set(nutrient["id"].dropna())
      ))

print("Duplicate food/nutrient pairs:",
      len(duplicate_pairs))

print("Missing sub-sample parents:",
      len(missing_parent))

print("Unmatched sub-sample results:",
      len(unmatched_results))


In [ ]:
print(food_nutrient.columns.tolist())
print(food_nutrient.head(10).to_string())
print(nutrient.columns.tolist())
print(nutrient.head(10).to_string())


In [ ]:
print(nutrient[nutrient["id"] == 2066])
print(food_nutrient[food_nutrient["nutrient_id"] == 2066].head(20))


In [ ]:
print(components.columns.tolist())
print(components.head(20).to_string())


In [ ]:
with open("food_component.csv", "r", encoding="utf-8-sig") as f:
    print(f.readline())


In [ ]:
missing_parents = set(sub_sample["fdc_id_of_sample_food"]) - set(food["fdc_id"])

print("Missing unique parent IDs:", len(missing_parents))
print("Missing parents:", sorted(missing_parents))


In [ ]:
print(food_nutrient_lab.columns.tolist())
print(food_nutrient_lab.head())


In [ ]:
print(food_nutrient.columns.tolist())
print(food_nutrient.head(10).to_string())
print(nutrient.head(10).to_string())


In [ ]:
print(components.columns.tolist())
print(components.head(10).to_string())


In [ ]:
print(sub_sample.columns.tolist())
print(sub_sample.head(10).to_string())


In [ ]:
%whos


In [ ]:
print(nutrient[nutrient["id"] == 2066])


In [ ]:
bad_2066 = food_nutrient[food_nutrient["nutrient_id"] == 2066]

print("Rows:", len(bad_2066))
print(bad_2066.to_string())


In [ ]:
components = pd.read_csv("food_component.csv")


In [ ]:
dupes = (
    food_nutrient
    .groupby(["fdc_id", "nutrient_id"])
    .size()
    .reset_index(name="count")
)

print(dupes[dupes["count"] > 1].to_string(index=False))


In [ ]:
for _, row in dupes[dupes["count"] > 1].iterrows():
    print("\n", row["fdc_id"], row["nutrient_id"])
    print(
        food_nutrient[
            (food_nutrient["fdc_id"] == row["fdc_id"]) &
            (food_nutrient["nutrient_id"] == row["nutrient_id"])
        ].to_string(index=False)
    )


In [ ]:
# Nutrients we actually care about
wanted_nutrients = [
    "Energy",
    "Protein",
    "Total lipid (fat)",
    "Carbohydrate, by difference",
    "Fiber, total dietary",
    "Sugars, total including NLEA",
    "Calcium, Ca",
    "Iron, Fe",
    "Magnesium, Mg",
    "Phosphorus, P",
    "Potassium, K",
    "Sodium, Na",
    "Vitamin C, total ascorbic acid",
    "Vitamin D (D2 + D3)",
    "Vitamin B-12",
    "Folate, total",
]

nutrient_lookup = nutrient[
    nutrient["name"].isin(wanted_nutrients)
].copy()

nutrient_lookup[
    ["id", "name", "unit_name", "nutrient_nbr", "rank"]
]


In [ ]:
# Join food nutrients to their nutrient definitions
food_nutrition = food_nutrient.merge(
    nutrient_lookup,
    left_on="nutrient_id",
    right_on="id",
    how="inner",
    suffixes=("_food_nutrient", "_nutrient")
)

# Join food descriptions
food_nutrition = food_nutrition.merge(
    food[["fdc_id", "description", "data_type"]],
    on="fdc_id",
    how="left"
)

food_nutrition[
    ["fdc_id", "description", "name", "amount", "unit_name", "data_type"]
].head(20)


In [ ]:
nutrition_wide = food_nutrition.pivot_table(
    index=["fdc_id", "description", "data_type"],
    columns="name",
    values="amount",
    aggfunc="first"
).reset_index()

nutrition_wide.columns.name = None

nutrition_wide.head()


In [ ]:
print(nutrition_wide.shape)

print(nutrition_wide.columns.tolist())

display(nutrition_wide.head())


In [ ]:
nutrition_wide[
    nutrition_wide["description"].str.contains(
        "apple", case=False, na=False
    )
].head(20)


In [ ]:
nutrition_wide[
    nutrition_wide["description"].str.contains(
        "chicken breast", case=False, na=False
    )
].head(20)


In [ ]:
print(nutrition_wide.shape)
print(nutrition_wide.columns.tolist())
display(nutrition_wide.head())


In [ ]:
# Build a searchable nutrient table directly from your existing data

search_data = (
    sub_sample_result
    .merge(
        nutrient_lookup[["id", "name", "unit_name"]],
        left_on="nutrient_id",
        right_on="id",
        how="left"
    )
    .merge(
        food[["fdc_id", "description"]].drop_duplicates("fdc_id"),
        on="fdc_id",
        how="left"
    )
)

search_data = search_data.rename(
    columns={
        "name": "nutrient_name"
    }
)

# Search for apple
apple = search_data[
    search_data["description"].str.contains(
        "apple", case=False, na=False
    )
]

print(
    apple[
        ["fdc_id", "description", "nutrient_name", "amount", "unit_name"]
    ].to_string(index=False)
)


In [ ]:
print("sub_sample_result columns:")
print(sub_sample_result.columns.tolist())

print("\nnutrient_lookup columns:")
print(nutrient_lookup.columns.tolist())

print("\nfood columns:")
print(food.columns.tolist())


In [ ]:
print("\nsub_sample_result:")
print(sub_sample_result.head().to_string(index=False))


In [ ]:
# Connect nutrient results → FDC food → food name

search_data = (
    sub_sample_result
    .merge(
        food_nutrient[["id", "fdc_id"]],
        left_on="food_nutrient_id",
        right_on="id",
        how="inner"
    )
    .merge(
        food[["fdc_id", "description"]],
        on="fdc_id",
        how="inner"
    )
)

# Search for Apple
apple = search_data[
    search_data["description"].str.contains(
        "apple", case=False, na=False
    )
].copy()

print(
    apple[
        ["fdc_id", "description", "nutrient_name", "adjusted_amount"]
    ].to_string(index=False)
)


In [ ]:
show_food("apple")


In [1]:
import pandas as pd

# 1. Load the USDA FoodData Central databases with the low_memory fix applied
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat (Total lipid), and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. Create the custom ID map for the Whole Foods
eight_whole_foods_id_map = {
    "Apple": 170279,          
    "Banana": 173944,         
    "Beef": 170208,           
    "Carrots": 170393,        
    "Chicken wings": 331897,  
    "Egg": 171287,            
    "Mushroom": 169251,       
    "Strawberries": 167762    
}

# 5. Extract just the nutrition rows for those specific foods
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(eight_whole_foods_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the FDC IDs back to the human-readable string names
reverse_food_map = {v: k for k, v in eight_whole_foods_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Leaving the variable at the bottom of the cell will render it as a clean HTML table in Jupyter
final_table

nutrient_name,"Carbohydrate, by difference",Protein,Total lipid (fat)
food,,,
Chicken wings,0.0,23.9,5.95


In [2]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Apple", "Banana", "Beef", "Carrots", "Chicken", "Egg", "Mushroom", "Strawberries"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

nutrient_name,"Carbohydrate, by difference",Protein,Total lipid (fat)
food,,,
Apple,14.8000,0.19,0.2100
Banana,20.1000,0.73,0.2200
Beef,2.8900,11.70,28.0000
Carrots,7.9200,0.81,0.4700
Chicken,0.0000,23.90,5.9500
Egg,0.9100,12.30,10.3000
Mushroom,7.5897,2.50,0.2563
Strawberries,7.6300,0.64,0.2200


In [ ]:
# Convert the pivot table to a nested dictionary for an API
nutrition_dict = final_table.to_dict(orient="index")
print(nutrition_dict)

In [ ]:
import pandas as pd

# 1. Load your USDA FoodData Central CSV files
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Get ALL Foundation Foods (Whole Foods)
foundation_foods = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. Filter and ADD .copy() to prevent the Pandas SettingWithCopyWarning
all_foundation_nutrition = food_nutrient[
    (food_nutrient["fdc_id"].isin(foundation_foods["fdc_id"])) & 
    (food_nutrient["nutrient_id"].isin(target_nutrient_ids))
].copy()

# 5. Map readable Nutrient Names
nutrient_map = target_nutrients.set_index("id")["name"].to_dict()
all_foundation_nutrition["nutrient_name"] = all_foundation_nutrition["nutrient_id"].map(nutrient_map)

# 6. Map readable Food Descriptions
food_map = foundation_foods.set_index("fdc_id")["description"].to_dict()
all_foundation_nutrition["food_description"] = all_foundation_nutrition["fdc_id"].map(food_map)

# 7. Pivot table to get ALL Foundation Foods as rows
all_foods_table = all_foundation_nutrition.pivot_table(
    index="food_description", 
    columns="nutrient_name", 
    values="amount"
).fillna(0.0)

# 8. Set Pandas options to show up to 400 rows so you can see all 361 items
pd.set_option('display.max_rows', 400)

# 9. Save to CSV
all_foods_table.to_csv("all_foundation_whole_foods.csv")

# Display the clean table
all_foods_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = [
  "Apple", "Banana", "Mango", "Orange", "Guava", "Papaya", "Pineapple",
  "Watermelon", "Muskmelon", "Pomegranate", "Grapes", "Pear", "Peach",
  "Plum", "Strawberries", "Blueberries", "Raspberries", "Cherries",
  "Coconut", "Dates", "Figs", "Raisins", "Jackfruit", "Custard Apple",
  "Sapota", "Lychee", "Dragon Fruit", "Kiwi", "Avocado",

  "Carrots", "Potatoes", "Sweet Potatoes", "Beetroot", "Radish",
  "Turnip", "Onion", "Garlic", "Ginger", "Tomato", "Cucumber",
  "Pumpkin", "Bottle Gourd", "Bitter Gourd", "Ridge Gourd",
  "Snake Gourd", "Ash Gourd", "Ivy Gourd", "Drumstick", "Okra",
  "Eggplant", "Green Peas", "Green Beans", "French Beans", "Corn",
  "Capsicum", "Green Chilli", "Cauliflower", "Broccoli", "Cabbage",
  "Spinach", "Amaranth", "Fenugreek Leaves", "Coriander", "Mint",
  "Curry Leaves", "Mushroom",

  "Lentils", "Red Lentils", "Green Lentils", "Black Lentils",
  "Yellow Lentils", "Chickpeas", "Black Chickpeas", "Kidney Beans",
  "Black Beans", "Green Gram", "Black Gram", "Pigeon Peas",
  "Cowpeas", "Soybeans", "Peanuts",

  "Rice", "Brown Rice", "White Rice", "Red Rice", "Black Rice",
  "Basmati Rice", "Oats", "Barley", "Millet", "Pearl Millet",
  "Finger Millet", "Sorghum", "Foxtail Millet", "Little Millet",
  "Barnyard Millet", "Kodo Millet", "Quinoa", "Whole Wheat",
  "Whole Wheat Flour", "Maize", "Buckwheat",

  "Almonds", "Cashews", "Walnuts", "Pistachios", "Pecans",
  "Hazelnuts", "Brazil Nuts", "Macadamia Nuts", "Pumpkin Seeds",
  "Sunflower Seeds", "Sesame Seeds", "Flax Seeds", "Chia Seeds",
  "Hemp Seeds",

  "Milk", "Curd", "Yogurt", "Paneer", "Cheese", "Eggs",
  "Chicken", "Turkey", "Beef", "Mutton", "Lamb", "Pork",
  "Fish", "Salmon", "Sardines", "Tuna", "Rohu", "Hilsa",
  "Prawns", "Crab",

  "Honey", "Fresh Coconut", "Coconut Milk"
]

dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Mango","Banana","Plantain","Guava","Papaya","Pomegranate","Grapes","Watermelon","Muskmelon","Pineapple","Coconut","Jackfruit","Custard Apple","Ramphal","Sapota","Lychee","Amla","Jamun","Bael","Wood Apple","Elephant Apple","Ber","Karonda","Kokum","Phalsa","Tamarind","Fig","Chikoo","Tadgola","Mahua","Kaitha","Kendu","Gular","Kafal","Hisalu","Kilmu","Buransh","Himalayan Raspberry","Himalayan Strawberry","Mulberry","Persimmon","Loquat","Peach","Plum","Apricot","Pear","Apple","Strawberry","Orange","Sweet Lime","Lemon","Mosambi","Mandarin","Kinnow","Pomelo","Grapefruit","Citron","Kagzi Lime","Galgal","Jamir","Narangi","Malta","Blood Orange","Dragon Fruit","Avocado","Passion Fruit","Tamarillo","Star Fruit","Bilimbi","Ambarella","Rose Apple","Wax Apple","Java Apple","Water Apple","Malay Apple","Rambutan","Mangosteen","Longan","Durian","Salak","Pulasan","Cempedak","Langsat","Duku","Santol","Breadfruit","Soursop","Star Apple","Canistel","Black Sapote","White Sapote","Mamey Sapote","Velvet Apple","Atemoya","Cherimoya","Snake Gourd Fruit","Camu Camu","Acerola","Jabuticaba","Feijoa","Lucuma","Cupuaçu","Cocona","Naranjilla","Pepino","Granadilla","Jamrul","Pilu","Jhar Ber","Jungle Jalebi","Kachnar Fruit","Chironji","Charoli","Pithraj","Pangam Fruit","Indian Persimmon","Khirni","Rasbhari","Cape Gooseberry","Ground Cherry","Lasoda","Lasora","Grewia Fruit","Gangren","Khirni","Makor","Khirni Ber","Aonla","Hog Plum","Indian Hog Plum","Amra","Ambada","Karamcha","Karamal","Karmal","Kundru Fruit","Tendu","Dhaman","Diospyros Melanoxylon Fruit","Mahua Fruit","Mahua Berry","Kusum Fruit","Palash Fruit","Buchanania Fruit","Agnimantha Fruit","Ritha Fruit","Haritaki","Bibhitaki","Harad","Baheda","Vibhitaki","Aak Fruit","Arjun Fruit","Neem Fruit","Moringa Fruit","Drumstick Fruit","Indian Gooseberry","Indian Jujube","Chinese Jujube","Wild Jujube","Desi Ber","Gokhru Fruit","Nagpur Orange","Coorg Orange","Darjeeling Orange","Khasi Mandarin","Sikkim Mandarin","Kodai Orange","Wam Orange","Sour Orange","Kumaon Lemon","Assam Lemon","Rangpur Lime","Kagzi Nimbu","Kachai Lemon","Sohiong","Sohphie","Sohshang","Sohiong Black Cherry","Sohphoh","Sohphie Plum","Sohiong Berry","Aiselu","Ainselu","Hisalu Berry","Timla","Bedu","Mehal","Kaafal","Kaphal","Kilmora","Kilmora Berry","Dadu","Daru","Daruharidra Fruit","Sea Buckthorn","Goji Berry","Wild Himalayan Cherry","Himalayan Wild Pear","Himalayan Wild Apricot","Himalayan Wild Plum","Himalayan Wild Peach","Himalayan Crab Apple","Crab Apple","Indian Crab Apple","Wild Fig","Cluster Fig","Indian Fig","Pakar Fruit","Banyan Fruit","Peepal Fruit","Indian Banyan Fig","Kadam Fruit","Jamun","Nerale Hannu","Nelli","Kundal Fruit","Narkel","Taad Fruit","Palmyra Fruit","Toddy Palm Fruit","Ice Apple","Borassus Fruit","Date Palm Fruit","Indian Date","Khejur","Taal Fruit","Phoenix Dactylifera Fruit","Areca Nut","Betel Nut","Wild Date","Doum Palm Fruit","Indian Wild Mango","Hog Plum","Wild Mango","Kokum Fruit","Kokum Berry","Myrica Fruit","Pistachio Fruit","Chironji Fruit","Cashew Apple","Cashew Fruit","Pineapple Guava","Indian Gooseberry Berry","Mango Ginger Fruit","Kokum Plum","Kundru Berry","Kharjura","Kharjur","Makhana Fruit","Lotus Fruit","Water Caltrop Fruit","Singhara","Fox Nut Fruit","Bakul Fruit","Mimusops Fruit","Nagkesar Fruit","Kadamba Fruit","Siris Fruit","Kachnar Fruit","Agnimantha Berry","Indian Laurel Fruit","Mahua Berry","Indian Laurel Cherry","Mimusops Elengi Fruit","Pangium Fruit","Wild Tamarind","Manila Tamarind","Madras Thorn","Jungle Jalebi Fruit","Kodukkapuli","Vilayati Imli","Tamarind Plum","Indian Plum","Allspice Berry","Pepper Fruit","Black Pepper Berry","Long Pepper Fruit"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Potato","Sweet Potato","Yam","Elephant Foot Yam","Greater Yam","Lesser Yam","Purple Yam","Chinese Yam","Taro","Colocasia","Cassava","Carrot","Radish","Turnip","Beetroot","Parsnip","Rutabaga","Onion","Shallot","Garlic","Ginger","Turmeric","Green Chilli","Red Chilli","Bell Pepper","Capsicum","Tomato","Brinjal","Eggplant","Okra","Drumstick","Bottle Gourd","Bitter Gourd","Ridge Gourd","Sponge Gourd","Snake Gourd","Ash Gourd","Pointed Gourd","Ivy Gourd","Round Gourd","Apple Gourd","Cucumber","Zucchini","Pumpkin","Chayote","Raw Banana","Raw Papaya","Raw Jackfruit","Green Mango","Green Peas","Snow Peas","French Beans","Cluster Beans","Broad Beans","Hyacinth Beans","Field Beans","Lima Beans","Cowpea","Black-eyed Peas","Soybean","Mung Bean","Urad Bean","Kidney Beans","Pigeon Peas","Chickpeas","Bengal Gram","Green Gram","Black Gram","Horse Gram","Lentils","Beet Greens","Amaranth Greens","Spinach","Malabar Spinach","Water Spinach","Fenugreek Leaves","Mustard Greens","Coriander Leaves","Mint Leaves","Curry Leaves","Dill Leaves","Parsley","Celery","Lettuce","Kale","Collard Greens","Swiss Chard","Bok Choy","Chinese Cabbage","Cabbage","Red Cabbage","Cauliflower","Broccoli","Brussels Sprouts","Kohlrabi","Radish Greens","Turnip Greens","Carrot Greens","Beet Greens","Mustard","Garden Cress","Watercress","Sorrel","Arugula","Leek","Spring Onion","Green Onion","Chives","Fennel Bulb","Artichoke","Asparagus","Bamboo Shoots","Lotus Root","Lotus Stem","Water Chestnut","Raw Banana Flower","Banana Stem","Banana Blossom","Drumstick Leaves","Drumstick Flowers","Agathi Leaves","Agathi Flowers","Moringa Leaves","Moringa Flowers","Neem Leaves","Neem Flowers","Sesbania Leaves","Sesbania Flowers","Roselle Leaves","Roselle Flowers","Colocasia Leaves","Colocasia Stems","Amaranth Stems","Amaranth Flowers","Pumpkin Leaves","Pumpkin Flowers","Bottle Gourd Leaves","Bottle Gourd Flowers","Ridge Gourd Leaves","Sweet Potato Leaves","Cassava Leaves","Tapioca","Tapioca Leaves","Yam Leaves","Elephant Foot Yam Leaves","Green Chickpeas","Fresh Pigeon Peas","Fresh Toor Dal","Fresh Lima Beans","Fresh Soybeans","Fresh Cowpeas","Fresh Broad Beans","Moth Beans","Matki","Field Peas","Black Gram Sprouts","Green Gram Sprouts","Bengal Gram Sprouts","Alfalfa Sprouts","Bamboo Shoot","Tender Jackfruit","Tender Tamarind","Raw Tamarind","Drumstick Pods","Mushroom","Button Mushroom","Oyster Mushroom","Shiitake Mushroom","Milky Mushroom","Paddy Straw Mushroom","Wood Ear Mushroom","Enoki Mushroom","Portobello Mushroom","Morel Mushroom","Termite Mushroom","Wild Mushroom","Corn","Baby Corn","Sweet Corn","Maize","Raw Coconut","Coconut Shoot","Avarekai","Avarakkai","Sem","Papdi","Valor Papdi","Guar","Gawar","Tendli","Tindora","Kundru","Parwal","Potol","Karela","Torai","Turai","Lauki","Dudhi","Tinda","Chichinda","Kovakkai","Peerkangai","Surakkai","Pudalangai","Poosanikai","Mathanga","Kaddu","Kumbalakai","Seemebadanekai","Chow Chow","Vazhakkai","Vazhaithandu","Vazhaipoo","Kathirikkai","Vendakkai","Murungakkai","Suran","Kachalu","Arbi","Shakarkand","Mooli","Gajar","Shalgam","Palak","Methi","Sarson","Bathua","Chaulai","Lal Saag","Poi Saag","Kolmi Saag","Nenua","Kachri","Ker","Sangri","Gunda","Kachnar Buds","Kachnar Flowers","Raw Lotus Seeds","Lotus Pods","Water Caltrop","Singhara","Makhana","Green Tamarind","Gongura","Khatta Palak","Manathakkali","Mudakathan","Pirandai","Vallarai","Thandu Keerai","Araikeerai","Sirukeerai","Pasalai Keerai","Ponnanganni Keerai","Agathi Keerai","Murungai Keerai","Keerai","Chukka Keerai","Cholai Keerai","Methi Matar","Green Garlic","Garlic Scapes","Onion Greens","Radish Pods","Mustard Pods","Drumstick Flowers","Neem Flowers","Banana Flower","Pumpkin Blossom","Squash Blossoms","Zucchini Blossoms","Artichoke Hearts","Baby Potato","New Potato","Fingerling Potato","Red Potato","Purple Potato","Raw Yam","Surti Papdi","Lilva","Tuvar Lilva","Green Chana","Matar","Matar Pods","Green Soybean","Edamame"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Rice","Brown Rice","White Rice","Red Rice","Black Rice","Purple Rice","Wild Rice","Basmati Rice","Sona Masuri Rice","Ponni Rice","Jeera Samba Rice","Seeraga Samba Rice","Kullakar Rice","Matta Rice","Kerala Red Rice","Navara Rice","Kalanamak Rice","Gobindobhog Rice","Ambemohar Rice","Indrayani Rice","Joha Rice","Katarni Rice","Bamboo Rice","Sticky Rice","Glutinous Rice","Parboiled Rice","Broken Rice","Wheat","Whole Wheat","Durum Wheat","Hard Red Wheat","Soft Wheat","Emmer Wheat","Spelt","Einkorn","Khorasan Wheat","Triticale","Rye","Barley","Pearl Barley","Hulled Barley","Naked Barley","Six-Row Barley","Two-Row Barley","Oats","Rolled Oats","Steel-Cut Oats","Millet","Pearl Millet","Finger Millet","Foxtail Millet","Little Millet","Kodo Millet","Barnyard Millet","Proso Millet","Browntop Millet","Japanese Millet","Sorghum","Jowar","Maize","Corn","Sweet Corn","Popcorn","Flint Corn","Dent Corn","Waxy Corn","White Corn","Yellow Corn","Blue Corn","Red Corn","Black Corn","Teff","Fonio","Quinoa","Amaranth","Buckwheat","Chia","Spelt","Farro","Freekeh","Einkorn","Rye","Triticale","Wild Rice","Job's Tears","Adlay","Canary Seed","Sago","Sago Pearls","Pearl Sago","Tapioca","Buckwheat Groats","Millet Groats","Oat Groats","Wheat Groats","Barley Groats","Rye Groats","Corn Grits","Wheat Germ","Wheat Bran","Rice Bran","Rice Germ","Barley Bran","Oat Bran","Sorghum Grain","Pearl Millet Grain","Finger Millet Grain","Foxtail Millet Grain","Little Millet Grain","Kodo Millet Grain","Barnyard Millet Grain","Proso Millet Grain","Browntop Millet Grain","Japanese Millet Grain","Indian Barnyard Millet","Indian Browntop Millet","Indian Kodo Millet","Indian Foxtail Millet","Indian Little Millet","Indian Proso Millet","Indian Pearl Millet","Indian Finger Millet","Indian Sorghum","Indian Maize","Indian Rice","Navara Rice","Kuthiraivali","Thinai","Samai","Varagu","Kambu","Ragi","Jowar","Bajra","Jhangora","Cheena","Kangni","Kutki","Kodra","Sama","Rajgira","Kuttu","Barley","Jau","Gehu","Cholam","Makka","Dhan","Basmati","Samba Rice","Mappillai Samba","Karunguruvai","Thooyamalli","Poongar Rice","Kichili Samba","Kattuyanam Rice","Seeraga Samba","Kichadi Samba","Arupatham Kuruvai","Illuppai Poo Samba","Ponni Rice","Kichadi Rice","Red Kavuni Rice","Black Kavuni Rice","Karuppu Kavuni","Mapillai Samba","Kattuyanam","Kullakar","Navara","Pokali Rice","Matta Rice","Wayanad Rice","Jeerakasala Rice","Gandhakasala Rice","Joha Rice","Bora Rice","Chokuwa Rice","Bao Rice","Gobindobhog","Katarni","Kalajeera Rice","Tulaipanji Rice","Radhunipagal Rice","Lal Dhan","Hansraj Rice","Ambemohar","Indrayani","Kolam Rice","Surti Kolam","HMT Rice","Sona Masuri","Ponni","Swarna Rice","MTU-1010","IR-64","IR-36","PR-14 Rice","PR-106 Rice","Sharbati Wheat","Lokwan Wheat","MP Wheat","Malwa Wheat","Durum Wheat","Emmer Wheat","Khapli Wheat","Bansi Wheat","Lok-1 Wheat","C-306 Wheat","Sujata Wheat","Kalyan Sona Wheat","HD-2967 Wheat","HD-3086 Wheat","PBW-343 Wheat","Desi Wheat","Indian Barley","Himalayan Barley","Naked Barley","Six-Row Barley","Two-Row Barley","Hulless Barley","Indian Oats","Indian Rye","Indian Sorghum","Indian Maize"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Get the exact nutrient IDs for Protein, Fat, and Carbohydrates
target_nutrients = nutrient[nutrient["name"].str.contains(
    "Protein|Total lipid|Carbohydrate, by difference", 
    case=False, 
    na=False
)]
target_nutrient_ids = target_nutrients["id"].tolist()

# 4. DYNAMICALLY search for the 2026 FDC IDs instead of hardcoding the 2021 IDs
search_foods = ["Chicken","Turkey","Duck","Goose","Quail","Pigeon","Guinea Fowl","Pheasant","Partridge","Ostrich","Emu","Rabbit","Hare","Lamb","Mutton","Goat Meat","Chevon","Beef","Veal","Buffalo Meat","Carabeef","Pork","Ham","Bacon","Venison","Deer Meat","Elk Meat","Moose Meat","Bison Meat","Yak Meat","Camel Meat","Horse Meat","Donkey Meat","Boar Meat","Wild Boar","Reindeer Meat","Antelope Meat","Gazelle Meat","Kangaroo Meat","Alligator Meat","Crocodile Meat","Frog Legs","Snail","Escargot","Liver","Chicken Liver","Beef Liver","Lamb Liver","Goat Liver","Pork Liver","Heart","Chicken Heart","Beef Heart","Lamb Heart","Goat Heart","Kidney","Chicken Kidney","Beef Kidney","Lamb Kidney","Goat Kidney","Brain","Tongue","Beef Tongue","Lamb Tongue","Sweetbreads","Tripe","Beef Tripe","Lamb Tripe","Goat Tripe","Oxtail","Beef Marrow","Bone Marrow","Chicken Gizzard","Duck Gizzard","Chicken Feet","Pork Feet","Pig Trotters","Lamb Shank","Beef Shank","Chicken Breast","Chicken Thigh","Chicken Drumstick","Chicken Wing","Turkey Breast","Duck Breast","Duck Leg","Lamb Chops","Lamb Ribs","Mutton Ribs","Goat Ribs","Pork Ribs","Pork Chops","Beef Steak","Beef Ribs","Beef Brisket","Beef Tenderloin","Beef Sirloin","Beef Chuck","Buffalo Ribs","Venison Steak","Rabbit Meat","Fish","Salmon","Atlantic Salmon","Pacific Salmon","Trout","Rainbow Trout","Brown Trout","Cod","Haddock","Pollock","Hake","Halibut","Tuna","Yellowfin Tuna","Bluefin Tuna","Skipjack Tuna","Mackerel","King Mackerel","Indian Mackerel","Sardine","Sardines","Anchovy","Herring","Shad","Hilsa","Rohu","Katla","Mrigal","Carp","Grass Carp","Silver Carp","Common Carp","Catfish","Walking Catfish","Pangasius","Tilapia","Pomfret","Silver Pomfret","Black Pomfret","Indian Butterfish","Sole","Flounder","Snapper","Red Snapper","Sea Bass","Barramundi","Grouper","Mahi Mahi","Swordfish","Marlin","Sailfish","Eel","Conger Eel","Stingray","Skate","Shark","Octopus","Squid","Cuttlefish","Prawn","Shrimp","Tiger Prawn","King Prawn","Crab","Mud Crab","Blue Crab","Snow Crab","Lobster","Rock Lobster","Spiny Lobster","Crayfish","Mussels","Clams","Oysters","Scallops","Cockles","Abalone","Whelks","Sea Urchin","Sea Cucumber","Roe","Fish Roe","Salmon Roe","Tobiko","Masago","Caviar","Eggs","Chicken Eggs","Duck Eggs","Goose Eggs","Quail Eggs","Turkey Eggs","Pigeon Eggs","Guinea Fowl Eggs","Ostrich Eggs","Fish Eggs","Milk","Cow Milk","Buffalo Milk","Goat Milk","Sheep Milk","Camel Milk","Donkey Milk","Yak Milk","Mare Milk","Full Cream Milk","Skim Milk","Buttermilk","Curd","Yogurt","Greek Yogurt","Kefir","Lassi","Chaas","Paneer","Cottage Cheese","Cheddar","Mozzarella","Parmesan","Gouda","Edam","Emmental","Swiss Cheese","Brie","Camembert","Feta","Ricotta","Mascarpone","Cream Cheese","Blue Cheese","Goat Cheese","Sheep Cheese","Buffalo Mozzarella","Processed Cheese","Cheese Curd","Milk Powder","Skimmed Milk Powder","Condensed Milk","Evaporated Milk","Cream","Heavy Cream","Whipping Cream","Sour Cream","Clotted Cream","Butter","Ghee","Clarified Butter","Whey","Whey Protein","Casein","Milk Solids","Milk Fat","Milk Skin","Khoya","Mawa","Rabri","Shrikhand","Basundi","Malai","Dahi","Mishti Doi","Chhena","Chhurpi","Khoa","Kulfi","Ice Cream","Gelato","Milkshake","Milk Tea","Milk Chocolate"]
dynamic_id_map = {}

for item in search_foods:
    # Find all foundation foods containing our search term
    matches = foundation_food[foundation_food["description"].str.contains(item, case=False, na=False)]
    if not matches.empty:
        # Grab the FDC ID of the first match and save it to our dictionary
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 5. Extract just the nutrition rows for our newly found 2026 FDC IDs
eight_whole_foods_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())]

# 6. Filter those rows to ONLY include the 3 target macronutrients
eight_whole_foods_df = eight_whole_foods_df[eight_whole_foods_df["nutrient_id"].isin(target_nutrient_ids)]

# 7. Map the readable Nutrient Names into the dataframe
eight_whole_foods_df["nutrient_name"] = eight_whole_foods_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 8. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
eight_whole_foods_df["food"] = eight_whole_foods_df["fdc_id"].map(reverse_food_map)

# 9. Pivot the table so Foods are rows and Nutrients are columns
final_table = eight_whole_foods_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# Display the final table
final_table

In [3]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Dynamic search with STRICT word boundaries (\b) to prevent false matches
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Drumstick", "Mushroom", "Oats"]
dynamic_id_map = {}

for item in search_foods:
    # \b ensures we only match the exact word, case-insensitive
    matches = foundation_food[foundation_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs (NO FILTERING for specific nutrients this time)
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# Display the final mega-table
final_table

nutrient_name,Ash,Beta-glucan,Biotin,"Calcium, Ca","Carbohydrate, by difference",Cholesterol,Citric acid,"Copper, Cu",Energy,Energy (Atwater General Factors),...,"Vitamin A, RAE",Vitamin B-12,Vitamin B-6,"Vitamin C, total ascorbic acid",Vitamin D (D2 + D3),"Vitamin D (D2 + D3), International Units",Vitamin D3 (cholecalciferol),Vitamin E (alpha-tocopherol),Water,"Zinc, Zn"
food,,,,,,,,,,,,,,,,,,,,,
Apple,0.1238,-,-,7.099,11.363962,-,0.0,0.003401,-,48.3763,...,-,-,0.01388,51.16,-,-,-,-,88.14,0.002125
Banana,0.3793,-,1.164,9.816,4.966175,-,-,0.061280,-,23.9398,...,-,-,0.28730,112.1,-,-,-,-,93.80,0.130400
Beef,2.7400,-,-,15.000,2.890000,-,-,0.046000,812.0,310.0000,...,3.0,0.97,0.13000,-,-,-,-,0.51,54.60,2.060000
Drumstick,0.9800,-,-,12.000,0.000000,127.0,-,0.063000,404.0,149.0000,...,7.0,0.41,0.37200,-,0.1,2.0,0.1,0.17,69.90,2.540000
Mushroom,1.0840,-,16.93,0.000,7.589700,-,-,0.177100,-,42.6655,...,-,-,0.06550,-,-,-,-,-,88.57,0.744800
Oats,1.7060,7.52,21.9,45.530,68.657550,-,-,0.427800,-,381.6260,...,-,-,0.13460,-,-,-,-,-,10.25,2.744000
Rye,1.4050,1.913,8.975,32.240,77.161800,-,-,0.338400,-,359.4000,...,-,-,0.16380,-,-,-,-,-,11.13,2.328000


In [ ]:
import pandas as pd
import json

# 1. Load the 2026 USDA FoodData Central databases
print("Loading USDA CSV files (this may take a moment)...")
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Dynamic search with STRICT word boundaries (\b) to prevent false matches
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Drumstick", "Mushroom", "Oats"]
dynamic_id_map = {}

print("Extracting and mapping foundation foods...")
for item in search_foods:
    # \b ensures we only match the exact word, case-insensitive
    matches = foundation_food[foundation_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs (NO FILTERING for specific nutrients)
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# 9. Convert the Pandas dataframe into a raw dictionary
raw_dict = final_table.to_dict(orient="index")

# 10. Re-structure the data into organized JSON for your API
api_payload = {}

for food_name, nutrients in raw_dict.items():
    api_payload[food_name] = {
        "Macros": {
            "Energy (kcal)": nutrients.get("Energy", " - "),
            "Protein (g)": nutrients.get("Protein", " - "),
            "Carbohydrates (g)": nutrients.get("Carbohydrate, by difference", " - "),
            "Total Fat (g)": nutrients.get("Total lipid (fat)", " - "),
            "Fiber (g)": nutrients.get("Fiber, total dietary", " - ")
        },
        "Vitamins": {
            "Vitamin A (RAE)": nutrients.get("Vitamin A, RAE", " - "),
            "Vitamin C (mg)": nutrients.get("Vitamin C, total ascorbic acid", " - "),
            "Vitamin D (IU)": nutrients.get("Vitamin D (D2 + D3), International Units", " - "),
            "Vitamin B-6 (mg)": nutrients.get("Vitamin B-6", " - ")
        },
        "Minerals": {
            "Calcium (mg)": nutrients.get("Calcium, Ca", " - "),
            "Zinc (mg)": nutrients.get("Zinc, Zn", " - "),
            "Copper (mg)": nutrients.get("Copper, Cu", " - ")
        },
        # Keeps all 100+ other nutrients available in the background
        "All_Nutrients": nutrients 
    }

# 11. Print the cleanly nested JSON 
print("\n=== FINAL JSON PAYLOAD ===")
print(json.dumps(api_payload, indent=2))

In [ ]:
"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv"

In [ ]:
import pandas as pd
import json

# 1. Load the USDA FoodData Central databases
print("Loading USDA CSV files (this may take a moment)...")
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food_nutrient.csv", low_memory=False)

# 2. Filter to "SR Legacy Foods" (This unlocks 7,000+ historical food entries)
sr_legacy_food = food[food["data_type"] == "sr_legacy_food"]

# 3. Dynamic search with STRICT word boundaries (\b) to prevent false matches
# I've included some standard whole foods as an example
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Drumstick", "Mushroom", "Oats"]
dynamic_id_map = {}

print("Extracting and mapping SR Legacy foods...")
for item in search_foods:
    # \b ensures we only match the exact word, case-insensitive
    matches = sr_legacy_food[sr_legacy_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        # Grabbing the first match found in the SR Legacy dataset
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# 9. Convert the Pandas dataframe into a raw dictionary
raw_dict = final_table.to_dict(orient="index")

# 10. Re-structure the data into organized JSON for your API
api_payload = {}

for food_name, nutrients in raw_dict.items():
    api_payload[food_name] = {
        "Macros": {
            "Energy (kcal)": nutrients.get("Energy", " - "),
            "Protein (g)": nutrients.get("Protein", " - "),
            "Carbohydrates (g)": nutrients.get("Carbohydrate, by difference", " - "),
            "Total Fat (g)": nutrients.get("Total lipid (fat)", " - "),
            "Fiber (g)": nutrients.get("Fiber, total dietary", " - ")
        },
        "Vitamins": {
            "Vitamin A (RAE)": nutrients.get("Vitamin A, RAE", " - "),
            "Vitamin C (mg)": nutrients.get("Vitamin C, total ascorbic acid", " - "),
            "Vitamin D (IU)": nutrients.get("Vitamin D (D2 + D3), International Units", " - "),
            "Vitamin B-6 (mg)": nutrients.get("Vitamin B-6", " - ")
        },
        "Minerals": {
            "Calcium (mg)": nutrients.get("Calcium, Ca", " - "),
            "Zinc (mg)": nutrients.get("Zinc, Zn", " - "),
            "Copper (mg)": nutrients.get("Copper, Cu", " - ")
        },
        # Keeps all other SR Legacy nutrients available in the background
        "All_Nutrients": nutrients 
    }

# 11. Print the cleanly nested JSON 
print("\n=== FINAL SR LEGACY JSON PAYLOAD ===")
print(json.dumps(api_payload, indent=2))

In [ ]:
import pandas as pd

# ================================================================
# 1. Load the USDA SR Legacy FoodData Central databases
# ================================================================

food = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv"
)

nutrient = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\nutrient.csv"
)

food_nutrient = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\food_nutrient.csv",
    low_memory=False
)


# ================================================================
# 2. Filter to "SR Legacy Foods"
# ================================================================

sr_legacy_food = food[
    food["data_type"] == "sr_legacy_food"
]


# ================================================================
# 3. Foods to search
# ================================================================

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]


# ================================================================
# 4. Dynamically find the FDC ID for each food
# ================================================================

dynamic_id_map = {}

for item in search_foods:

    # \b = strict word boundary
    # case=False = case-insensitive
    # na=False = ignore missing descriptions

    matches = sr_legacy_food[
        sr_legacy_food["description"].str.contains(
            rf"\b{item}\b",
            case=False,
            na=False,
            regex=True
        )
    ]

    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]


# ================================================================
# 5. Extract ALL nutrition rows for the selected foods
# ================================================================

all_nutrients_df = food_nutrient[
    food_nutrient["fdc_id"].isin(
        dynamic_id_map.values()
    )
].copy()


# ================================================================
# 6. Map nutrient IDs to readable nutrient names
# ================================================================

all_nutrients_df["nutrient_name"] = (
    all_nutrients_df["nutrient_id"].map(
        nutrient.set_index("id")["name"]
    )
)


# ================================================================
# 7. Map FDC IDs back to food names
# ================================================================

reverse_food_map = {
    fdc_id: food_name
    for food_name, fdc_id in dynamic_id_map.items()
}

all_nutrients_df["food"] = (
    all_nutrients_df["fdc_id"].map(
        reverse_food_map
    )
)


# ================================================================
# 8. Pivot the table
#
# Food = rows
# Nutrients = columns
# Amount = values
# ================================================================

final_table = all_nutrients_df.pivot_table(
    index="food",
    columns="nutrient_name",
    values="amount"
)


# ================================================================
# 9. Replace missing values with " - "
# ================================================================

final_table = final_table.fillna(" - ")


# ================================================================
# 10. Put the important nutrients first
#     Everything else remains available after them
# ================================================================

preferred_columns = [
    "Energy",
    "Protein",
    "Carbohydrate, by difference",
    "Total lipid (fat)",
    "Fiber, total dietary",

    "Vitamin A, RAE",
    "Vitamin C, total ascorbic acid",
    "Vitamin D (D2 + D3), International Units",
    "Vitamin B-6",

    "Calcium, Ca",
    "Zinc, Zn",
    "Copper, Cu"
]


existing_preferred = [
    column
    for column in preferred_columns
    if column in final_table.columns
]

remaining_columns = [
    column
    for column in final_table.columns
    if column not in existing_preferred
]

final_table = final_table[
    existing_preferred + remaining_columns
]


# ================================================================
# 11. Display the final mega-table
# ================================================================

print("\n=== USDA SR LEGACY NUTRITION TABLE ===\n")

final_table


In [ ]:
import pandas as pd

# 1. Load the USDA FoodData Central databases
print("Loading USDA CSV files (this may take a moment)...")
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04\FoodData_Central_sr_legacy_food_csv_2018-04\food_nutrient.csv", low_memory=False)

# 2. Filter to "SR Legacy Foods" 
sr_legacy_food = food[food["data_type"] == "sr_legacy_food"]

# 3. Dynamic search with STRICT word boundaries (\b)
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Mushroom", "Oats"]
dynamic_id_map = {}

print("Extracting and mapping SR Legacy foods...")
for item in search_foods:
    matches = sr_legacy_food[sr_legacy_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# 9. Define the primary columns you want to display in the console table
display_columns = [
    "Energy", 
    "Protein", 
    "Carbohydrate, by difference", 
    "Total lipid (fat)", 
    "Fiber, total dietary"
]

# Ensure we only try to display columns that actually exist in the result
valid_columns = [col for col in display_columns if col in final_table.columns]
display_table = final_table[valid_columns].copy()

# Rename the columns so they look much cleaner in the console
display_table.columns = ["Calories", "Protein (g)", "Carbs (g)", "Fat (g)", "Fiber (g)"]

# 10. Print the clean table to the terminal
print("\n" + "="*65)
print(" SR LEGACY MACROS (Per 100g)")
print("="*65)
print(display_table.to_string())
print("="*65)

# 11. Export the FULL 121-column dataset to Excel/CSV for your records
export_path = "sr_legacy_extracted_data.csv"
final_table.to_csv(export_path)
print(f"\n[+] Full dataset with all vitamins and minerals saved to: {export_path}")

In [ ]:
import pandas as pd

# 1. Load the 2026 USDA FoodData Central databases
food = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food.csv")
nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\nutrient.csv")
food_nutrient = pd.read_csv(r"C:\Users\AK\Downloads\zip\FoodData_Central_foundation_food_csv_2026-04-30\FoodData_Central_foundation_food_csv_2026-04-30\food_nutrient.csv", low_memory=False)

# 2. Filter to "Foundation Foods" 
foundation_food = food[food["data_type"] == "foundation_food"]

# 3. Dynamic search with STRICT word boundaries (\b) to prevent false matches
search_foods = ["Apple", "Banana", "Beef", "Chicken", "Rye", "Drumstick", "Mushroom", "Oats"]
dynamic_id_map = {}

for item in search_foods:
    # \b ensures we only match the exact word, case-insensitive
    matches = foundation_food[foundation_food["description"].str.contains(rf"\b{item}\b", case=False, na=False, regex=True)]
    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

# 4. Extract nutrition rows for the found IDs (NO FILTERING for specific nutrients this time)
all_nutrients_df = food_nutrient[food_nutrient["fdc_id"].isin(dynamic_id_map.values())].copy()

# 5. Map the readable Nutrient Names into the dataframe
all_nutrients_df["nutrient_name"] = all_nutrients_df["nutrient_id"].map(
    nutrient.set_index("id")["name"]
)

# 6. Reverse map the new FDC IDs back to our human-readable names
reverse_food_map = {v: k for k, v in dynamic_id_map.items()}
all_nutrients_df["food"] = all_nutrients_df["fdc_id"].map(reverse_food_map)

# 7. Pivot the table so Foods are rows and ALL available Nutrients become columns
final_table = all_nutrients_df.pivot_table(
    index="food", 
    columns="nutrient_name", 
    values="amount"
)

# 8. Replace all NaN (missing) values with a hyphen ' - '
final_table = final_table.fillna(' - ')

# Display the final mega-table
final_table

In [ ]:
import pandas as pd
import json


# ================================================================
# 1. LOAD USDA FOODDATA CENTRAL DATABASES
# ================================================================

print("Loading USDA CSV files (this may take a moment)...")

BASE_PATH = (
    r"C:\Users\AK\Downloads\zip"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04"
)

food = pd.read_csv(
    BASE_PATH + r"\food.csv"
)

nutrient = pd.read_csv(
    BASE_PATH + r"\nutrient.csv"
)

food_nutrient = pd.read_csv(
    BASE_PATH + r"\food_nutrient.csv",
    low_memory=False
)


# ================================================================
# 2. FILTER SR LEGACY FOODS
# ================================================================

sr_legacy_food = food[
    food["data_type"] == "sr_legacy_food"
]


# ================================================================
# 3. FOODS TO SEARCH
# ================================================================

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]


# ================================================================
# 4. FIND FDC IDs FOR THE FOODS
# ================================================================

dynamic_id_map = {}

print("Extracting and mapping SR Legacy foods...")

for item in search_foods:

    matches = sr_legacy_food[
        sr_legacy_food["description"].str.contains(
            rf"\b{item}\b",
            case=False,
            na=False,
            regex=True
        )
    ]

    if not matches.empty:

        # Take the first matching food
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]

    else:

        print(f"WARNING: {item} was not found.")


# ================================================================
# 5. EXTRACT NUTRIENTS FOR FOUND FOODS
# ================================================================

all_nutrients_df = food_nutrient[
    food_nutrient["fdc_id"].isin(
        dynamic_id_map.values()
    )
].copy()


# ================================================================
# 6. MAP NUTRIENT ID -> NUTRIENT NAME
# ================================================================

nutrient_name_map = nutrient.set_index("id")["name"]

all_nutrients_df["nutrient_name"] = (
    all_nutrients_df["nutrient_id"]
    .map(nutrient_name_map)
)


# ================================================================
# 7. MAP FDC ID -> FOOD NAME
# ================================================================

reverse_food_map = {
    fdc_id: food_name
    for food_name, fdc_id in dynamic_id_map.items()
}

all_nutrients_df["food"] = (
    all_nutrients_df["fdc_id"]
    .map(reverse_food_map)
)


# ================================================================
# 8. CREATE FOOD x NUTRIENT TABLE
#
#     Food       Energy   Protein   Carbs   Fat   ...
#     Apple      752.5     2.17    44.54  11.5
#     Banana     273.0     2.76    19.74   1.7
#
# ================================================================

final_table = all_nutrients_df.pivot_table(
    index="food",
    columns="nutrient_name",
    values="amount",
    aggfunc="first"
)


# ================================================================
# 9. RENAME IMPORTANT NUTRIENTS
#
# This makes the column names shorter and easier to read.
# ================================================================

rename_columns = {
    "Energy": "Energy",
    "Protein": "Protein",
    "Carbohydrate, by difference": "Carbs",
    "Total lipid (fat)": "Fat",
    "Fiber, total dietary": "Fiber",

    "Vitamin A, RAE": "Vitamin A",
    "Vitamin C, total ascorbic acid": "Vitamin C",
    "Vitamin D (D2 + D3), International Units": "Vitamin D",
    "Vitamin B-6": "Vitamin B6",

    "Calcium, Ca": "Calcium",
    "Zinc, Zn": "Zinc",
    "Copper, Cu": "Copper"
}

final_table = final_table.rename(
    columns=rename_columns
)


# ================================================================
# 10. DEFINE THE IMPORTANT COLUMN ORDER
# ================================================================

main_columns = [
    "Energy",
    "Protein",
    "Carbs",
    "Fat",
    "Fiber",

    "Vitamin A",
    "Vitamin C",
    "Vitamin D",
    "Vitamin B6",

    "Calcium",
    "Zinc",
    "Copper"
]


# ================================================================
# 11. PUT IMPORTANT NUTRIENTS FIRST
#
# Everything else comes after them.
# ================================================================

existing_main_columns = [
    column
    for column in main_columns
    if column in final_table.columns
]

other_columns = [
    column
    for column in final_table.columns
    if column not in existing_main_columns
]

final_table = final_table[
    existing_main_columns + other_columns
]


# ================================================================
# 12. REPLACE MISSING VALUES
# ================================================================

final_table = final_table.fillna(" - ")


# ================================================================
# 13. FORMAT NUMBERS
# ================================================================

def format_value(value):

    # Missing value
    if pd.isna(value):
        return " - "

    # Numeric value
    if isinstance(value, (int, float)):
        return f"{value:.2f}"

    # String
    return str(value)


final_table = final_table.map(format_value)


# ================================================================
# 14. PRINT FINAL TABLE
# ================================================================

print("\n")
print("=" * 200)

print("                         USDA NUTRITION TABLE")

print("=" * 200)

print(
    final_table.to_string(
        justify="right"
    )
)

print("=" * 200)


# ================================================================
# 15. OPTIONAL: SAVE TABLE TO CSV
# ================================================================

final_table.to_csv(
    "USDA_nutrition_table.csv"
)

print("\nTable saved to:")
print("USDA_nutrition_table.csv")


In [ ]:
import pandas as pd

# ================================================================
# 1. Load the USDA SR Legacy FoodData Central databases
# ================================================================

food = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\food.csv"
)

nutrient = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\nutrient.csv"
)

food_nutrient = pd.read_csv(
    r"C:\Users\AK\Downloads\zip\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04\food_nutrient.csv",
    low_memory=False
)


# ================================================================
# 2. Filter to "SR Legacy Foods"
# ================================================================

sr_legacy_food = food[
    food["data_type"] == "sr_legacy_food"
]


# ================================================================
# 3. Foods to search
# ================================================================

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]


# ================================================================
# 4. Dynamically find the FDC ID for each food
# ================================================================

dynamic_id_map = {}

for item in search_foods:

    # \b = strict word boundary
    # case=False = case-insensitive
    # na=False = ignore missing descriptions

    matches = sr_legacy_food[
        sr_legacy_food["description"].str.contains(
            rf"\b{item}\b",
            case=False,
            na=False,
            regex=True
        )
    ]

    if not matches.empty:
        dynamic_id_map[item] = matches.iloc[0]["fdc_id"]


# ================================================================
# 5. Extract ALL nutrition rows for the selected foods
# ================================================================

all_nutrients_df = food_nutrient[
    food_nutrient["fdc_id"].isin(
        dynamic_id_map.values()
    )
].copy()


# ================================================================
# 6. Map nutrient IDs to readable nutrient names
# ================================================================

all_nutrients_df["nutrient_name"] = (
    all_nutrients_df["nutrient_id"].map(
        nutrient.set_index("id")["name"]
    )
)


# ================================================================
# 7. Map FDC IDs back to food names
# ================================================================

reverse_food_map = {
    fdc_id: food_name
    for food_name, fdc_id in dynamic_id_map.items()
}

all_nutrients_df["food"] = (
    all_nutrients_df["fdc_id"].map(
        reverse_food_map
    )
)


# ================================================================
# 8. Pivot the table
#
# Food = rows
# Nutrients = columns
# Amount = values
# ================================================================

final_table = all_nutrients_df.pivot_table(
    index="food",
    columns="nutrient_name",
    values="amount"
)


# ================================================================
# 9. Replace missing values with " - "
# ================================================================

final_table = final_table.fillna(" - ")


# ================================================================
# 10. Put the important nutrients first
#     Everything else remains available after them
# ================================================================

preferred_columns = [
    "Energy",
    "Protein",
    "Carbohydrate, by difference",
    "Total lipid (fat)",
    "Fiber, total dietary",

    "Vitamin A, RAE",
    "Vitamin C, total ascorbic acid",
    "Vitamin D (D2 + D3), International Units",
    "Vitamin B-6",

    "Calcium, Ca",
    "Zinc, Zn",
    "Copper, Cu"
]


existing_preferred = [
    column
    for column in preferred_columns
    if column in final_table.columns
]

remaining_columns = [
    column
    for column in final_table.columns
    if column not in existing_preferred
]

final_table = final_table[
    existing_preferred + remaining_columns
]


# ================================================================
# 11. Display the final mega-table
# ================================================================

print("\n=== USDA SR LEGACY NUTRITION TABLE ===\n")

final_table


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)


In [ ]:
USDA_DIR = Path(
    r"C:\Users\AK\Downloads\zip"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04"
    r"\FoodData_Central_sr_legacy_food_csv_2018-04"
)

USDA_DIR


In [ ]:
food = pd.read_csv(
    USDA_DIR / "food.csv",
    low_memory=False
)

nutrient = pd.read_csv(
    USDA_DIR / "nutrient.csv",
    low_memory=False
)

food_nutrient = pd.read_csv(
    USDA_DIR / "food_nutrient.csv",
    low_memory=False
)

food_portion = pd.read_csv(
    USDA_DIR / "food_portion.csv",
    low_memory=False
)

measure_unit = pd.read_csv(
    USDA_DIR / "measure_unit.csv",
    low_memory=False
)


In [ ]:
print("food:", food.shape)
print("nutrient:", nutrient.shape)
print("food_nutrient:", food_nutrient.shape)
print("food_portion:", food_portion.shape)
print("measure_unit:", measure_unit.shape)

In [ ]:
print("FOOD")
print(food.columns.tolist())

print("\nNUTRIENT")
print(nutrient.columns.tolist())

print("\nFOOD_NUTRIENT")
print(food_nutrient.columns.tolist())

print("\nFOOD_PORTION")
print(food_portion.columns.tolist())

print("\nMEASURE_UNIT")
print(measure_unit.columns.tolist())


In [ ]:
sr_food = food[
    food["data_type"].eq("sr_legacy_food")
].copy()

print("Total foods:", len(food))
print("SR Legacy foods:", len(sr_food))


In [ ]:
sr_food[
    ["fdc_id", "data_type", "description"]
].head(20)


In [ ]:
def search_food(query, limit=20):
    
    query = query.strip()
    
    results = sr_food[
        sr_food["description"].str.contains(
            query,
            case=False,
            na=False,
            regex=False
        )
    ].copy()
    
    return results[
        [
            "fdc_id",
            "description",
            "data_type"
        ]
    ].head(limit)


In [ ]:
search_food("apple", 20)

In [ ]:
search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]

for item in search_foods:
    
    print("\n" + "=" * 80)
    print(item)
    print("=" * 80)
    
    display(
        search_food(item, 10)
    )

In [ ]:
selected_fdc_ids = list(selected_foods.values())

selected_food_df = sr_food[
    sr_food["fdc_id"].isin(selected_fdc_ids)
].copy()

selected_food_df[
    [
        "fdc_id",
        "description",
        "data_type"
    ]
]

In [4]:
import pandas as pd
import re

# ---------------------------------------------------------
# 1. Foods you want to investigate
# ---------------------------------------------------------

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats"
]


# ---------------------------------------------------------
# 2. Function to find ALL Foundation Food matches
# ---------------------------------------------------------

def find_food_matches(query, foundation_food):
    """
    Search USDA Foundation Foods and return all matching foods.
    """

    # Escape the query so special regex characters don't cause problems
    escaped_query = re.escape(query)

    matches = foundation_food[
        foundation_food["description"].str.contains(
            escaped_query,
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    # Keep the useful columns
    columns = ["fdc_id", "description"]

    # Only use columns that actually exist
    columns = [col for col in columns if col in matches.columns]

    return matches[columns].sort_values("description")


# ---------------------------------------------------------
# 3. Inspect every search term
# ---------------------------------------------------------

for item in search_foods:

    matches = find_food_matches(item, foundation_food)

    print("\n" + "=" * 80)
    print(f"SEARCH: {item}")
    print("=" * 80)

    if matches.empty:
        print("NO MATCHES FOUND")
    else:
        print(f"Found {len(matches)} matches:\n")
        print(matches.to_string(index=False))


SEARCH: Apple
Found 14 matches:

 fdc_id                                                       description
2003590 Apple juice, with added vitamin C, from concentrate, shelf stable
1105897                                      Apples, fuji, with skin, raw
1750340                                      Apples, fuji, with skin, raw
1105781                                      Apples, gala, with skin, raw
1750341                                      Apples, gala, with skin, raw
1105664                              Apples, granny smith, with skin, raw
1750342                              Apples, granny smith, with skin, raw
1105547                                Apples, honeycrisp, with skin, raw
1750343                                Apples, honeycrisp, with skin, raw
1105430                             Apples, red delicious, with skin, raw
1750339                             Apples, red delicious, with skin, raw
2263892                     Applesauce, unsweetened, with added vitamin C
2346

In [5]:
import pandas as pd
import re


# =========================================================
# 1. Function to search Foundation Foods
# =========================================================

def search_foundation_food(query, foundation_food):
    """
    Return all Foundation Foods whose description contains
    the search query.
    """

    query = str(query).strip()

    if not query:
        return pd.DataFrame(columns=["fdc_id", "description"])

    matches = foundation_food[
        foundation_food["description"].str.contains(
            re.escape(query),
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    return matches[["fdc_id", "description"]].sort_values("description")


# =========================================================
# 2. Function to find the best matching food
# =========================================================

def select_best_food(query, foundation_food, required_terms=None):
    """
    Find the best Foundation Food for a query.

    required_terms:
        Words/phrases that should ideally appear in the
        USDA description.

    Returns:
        selected food, confidence status, and all candidates.
    """

    matches = search_foundation_food(query, foundation_food)

    if matches.empty:
        return {
            "status": "NOT_FOUND",
            "selected": None,
            "candidates": matches
        }

    # If no extra requirements were provided,
    # return candidates for inspection.
    if not required_terms:
        return {
            "status": "REVIEW",
            "selected": None,
            "candidates": matches
        }

    # -----------------------------------------------------
    # Score every candidate
    # -----------------------------------------------------

    scored = []

    for _, row in matches.iterrows():

        description = row["description"].lower()

        score = 0

        for term in required_terms:

            term = term.lower().strip()

            if term in description:
                score += 1

        scored.append({
            "fdc_id": row["fdc_id"],
            "description": row["description"],
            "score": score
        })

    scored_df = pd.DataFrame(scored)

    scored_df = scored_df.sort_values(
        ["score", "description"],
        ascending=[False, True]
    ).reset_index(drop=True)

    best = scored_df.iloc[0]

    # -----------------------------------------------------
    # Decide whether selection is safe
    # -----------------------------------------------------

    if best["score"] == len(required_terms):

        # Check whether another food has the same score
        top_matches = scored_df[
            scored_df["score"] == best["score"]
        ]

        if len(top_matches) == 1:
            status = "SELECTED"
            selected = best
        else:
            status = "AMBIGUOUS"
            selected = None

    else:
        status = "REVIEW"
        selected = None

    return {
        "status": status,
        "selected": selected,
        "candidates": scored_df
    }

In [6]:
# ============================================================
# USDA FOODDATA CENTRAL FOUNDATION FOODS PIPELINE
# ============================================================

import pandas as pd
import re
from pathlib import Path


# ============================================================
# 1. FILE LOCATIONS
# ============================================================
# CHANGE THESE PATHS to match your files.

DATA_FOLDER = Path(".")

FOOD_CSV = DATA_FOLDER / "food.csv"
FOOD_NUTRIENT_CSV = DATA_FOLDER / "food_nutrient.csv"
NUTRIENT_CSV = DATA_FOLDER / "nutrient.csv"


# ============================================================
# 2. LOAD USDA DATA
# ============================================================

print("Loading USDA files...")

food = pd.read_csv(FOOD_CSV, low_memory=False)
food_nutrient = pd.read_csv(FOOD_NUTRIENT_CSV, low_memory=False)
nutrient = pd.read_csv(NUTRIENT_CSV, low_memory=False)

print("Food rows:", len(food))
print("Food nutrient rows:", len(food_nutrient))
print("Nutrient definitions:", len(nutrient))


# ============================================================
# 3. CHECK COLUMNS
# ============================================================

print("\nFood columns:")
print(food.columns.tolist())

print("\nFood nutrient columns:")
print(food_nutrient.columns.tolist())

print("\nNutrient columns:")
print(nutrient.columns.tolist())


# ============================================================
# 4. KEEP FOUNDATION FOODS ONLY
# ============================================================

foundation_food = food[
    food["data_type"]
    .astype(str)
    .str.lower()
    .eq("foundation_food")
].copy()

print("\nFoundation Foods:", len(foundation_food))


# ============================================================
# 5. CLEAN DESCRIPTION
# ============================================================

foundation_food["description"] = (
    foundation_food["description"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 6. SEARCH FUNCTION
# ============================================================

def search_foundation_food(query):
    """
    Search Foundation Foods and return all candidates.
    """

    query = str(query).strip()

    if not query:
        return pd.DataFrame(
            columns=["fdc_id", "description"]
        )

    pattern = re.escape(query)

    matches = foundation_food[
        foundation_food["description"].str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    return matches[
        ["fdc_id", "description"]
    ].sort_values("description")


# ============================================================
# 7. INSPECT YOUR SEARCH TERMS
# ============================================================

search_foods = [
    "Apple",
    "Banana",
    "Beef",
    "Chicken",
    "Rye",
    "Drumstick",
    "Mushroom",
    "Oats",
    "Carrot",
    "Milk"
]

print("\n\n")
print("=" * 80)
print("FOUNDATION FOOD SEARCH")
print("=" * 80)

for item in search_foods:

    matches = search_foundation_food(item)

    print("\n" + "-" * 80)
    print("SEARCH:", item)
    print("-" * 80)

    if matches.empty:
        print("NO MATCHES FOUND")
    else:
        print("Matches:", len(matches))
        print(matches.to_string(index=False))


# ============================================================
# 8. DEFINE THE SPECIFIC FOODS WE WANT
# ============================================================
#
# These are TARGET DESCRIPTIONS.
#
# USDA's exact wording may differ in your 2026 dataset.
# Therefore, inspect the output from section 7 first and
# adjust these terms if necessary.
#

target_foods = {

    "apple": {
        "search": "Apple",
        "required_terms": ["raw"]
    },

    "banana": {
        "search": "Banana",
        "required_terms": ["raw"]
    },

    "chicken_breast": {
        "search": "Chicken",
        "required_terms": ["breast", "raw"]
    },

    "beef_ground": {
        "search": "Beef",
        "required_terms": ["ground", "raw"]
    },

    "oats": {
        "search": "Oats",
        "required_terms": ["raw"]
    },

    "carrot": {
        "search": "Carrot",
        "required_terms": ["raw"]
    },

    "milk": {
        "search": "Milk",
        "required_terms": ["whole"]
    }
}


# ============================================================
# 9. FOOD SELECTION FUNCTION
# ============================================================

def select_food(search, required_terms):
    """
    Search Foundation Foods and score candidates.

    A food is automatically selected only when:
    - all required terms are present
    - there is only one top-scoring candidate

    Otherwise it is sent for manual review.
    """

    matches = search_foundation_food(search)

    if matches.empty:

        return {
            "status": "NOT_FOUND",
            "selected": None,
            "candidates": matches
        }

    results = []

    for _, row in matches.iterrows():

        description = row["description"].lower()

        score = 0

        for term in required_terms:

            if term.lower() in description:
                score += 1

        results.append({
            "fdc_id": row["fdc_id"],
            "description": row["description"],
            "score": score
        })

    candidates = pd.DataFrame(results)

    candidates = candidates.sort_values(
        ["score", "description"],
        ascending=[False, True]
    ).reset_index(drop=True)

    best_score = candidates.iloc[0]["score"]

    top = candidates[
        candidates["score"] == best_score
    ]

    # --------------------------------------------------------
    # Automatically select ONLY when there is one clear
    # candidate containing every required term.
    # --------------------------------------------------------

    if (
        best_score == len(required_terms)
        and len(top) == 1
    ):

        return {
            "status": "SELECTED",
            "selected": top.iloc[0],
            "candidates": candidates
        }

    elif len(top) > 1:

        return {
            "status": "AMBIGUOUS",
            "selected": None,
            "candidates": candidates
        }

    else:

        return {
            "status": "REVIEW",
            "selected": None,
            "candidates": candidates
        }


# ============================================================
# 10. SELECT FOODS
# ============================================================

selected_foods = {}
review_foods = {}

print("\n\n")
print("=" * 80)
print("SELECTING FOUNDATION FOODS")
print("=" * 80)

for food_name, config in target_foods.items():

    result = select_food(
        config["search"],
        config["required_terms"]
    )

    print("\n" + "-" * 80)
    print(food_name.upper())
    print("-" * 80)

    print("STATUS:", result["status"])

    if result["status"] == "SELECTED":

        selected = result["selected"]

        selected_foods[food_name] = {
            "fdc_id": int(selected["fdc_id"]),
            "description": selected["description"]
        }

        print("FDC ID:", selected["fdc_id"])
        print("Description:", selected["description"])

    else:

        review_foods[food_name] = result["candidates"]

        print("This food requires review.")

        print(
            result["candidates"]
            .head(20)
            .to_string(index=False)
        )


# ============================================================
# 11. SHOW SELECTED FOODS
# ============================================================

print("\n\n")
print("=" * 80)
print("SELECTED FOODS")
print("=" * 80)

for food_name, data in selected_foods.items():

    print(
        f"{food_name:20} "
        f"{data['fdc_id']:10} "
        f"{data['description']}"
    )


# ============================================================
# 12. BUILD FDC ID MAP
# ============================================================

dynamic_id_map = {
    name: data["fdc_id"]
    for name, data in selected_foods.items()
}

print("\nFDC ID MAP:")
print(dynamic_id_map)


# ============================================================
# 13. IDENTIFY NUTRIENT COLUMNS
# ============================================================

print("\n\n")
print("=" * 80)
print("NUTRIENT TABLE INSPECTION")
print("=" * 80)

print(nutrient.head())


# ============================================================
# 14. FIND MACRONUTRIENT DEFINITIONS
# ============================================================

nutrient_name_column = None

for column in ["name", "nutrient_name"]:

    if column in nutrient.columns:
        nutrient_name_column = column
        break

if nutrient_name_column is None:

    raise ValueError(
        "Could not find nutrient name column."
    )


macro_definitions = nutrient[
    nutrient[nutrient_name_column]
    .astype(str)
    .str.contains(
        "protein|total lipid|carbohydrate",
        case=False,
        na=False,
        regex=True
    )
].copy()

print("\nPotential macronutrients:")
print(macro_definitions.to_string(index=False))


# ============================================================
# 15. FIND THE CORRECT USDA NUTRIENT IDs
# ============================================================

print("\n")
print("IMPORTANT:")
print("Use the output above to verify the nutrient IDs.")
print("Do NOT blindly assume IDs if your USDA file differs.")


# ============================================================
# 16. COMMON USDA MACRO IDS
# ============================================================
#
# USDA commonly uses:
#
# 1003 = Protein
# 1004 = Total lipid (fat)
# 1005 = Carbohydrate, by difference
#
# We verify that they exist in YOUR nutrient table.
#

COMMON_MACRO_IDS = {
    "protein_g": 1003,
    "fat_g": 1004,
    "carbohydrate_g": 1005
}


available_nutrient_ids = set(
    nutrient["id"].dropna().astype(int)
)

nutrient_ids = {}

for name, nutrient_id in COMMON_MACRO_IDS.items():

    if nutrient_id in available_nutrient_ids:

        nutrient_ids[name] = nutrient_id

    else:

        print(
            f"WARNING: nutrient ID {nutrient_id} "
            f"not found for {name}"
        )


print("\nUsing nutrient IDs:")
print(nutrient_ids)


# ============================================================
# 17. FIND THE FDC ID COLUMN
# ============================================================

if "fdc_id" not in food_nutrient.columns:

    raise ValueError(
        "food_nutrient.csv does not contain fdc_id."
    )


# ============================================================
# 18. FIND THE NUTRIENT ID COLUMN
# ============================================================

if "nutrient_id" not in food_nutrient.columns:

    raise ValueError(
        "food_nutrient.csv does not contain nutrient_id."
    )


# ============================================================
# 19. FIND THE AMOUNT COLUMN
# ============================================================

amount_column = None

for column in ["amount", "nutrient_amount"]:

    if column in food_nutrient.columns:
        amount_column = column
        break

if amount_column is None:

    raise ValueError(
        "Could not find nutrient amount column."
    )


print("\nNutrient amount column:", amount_column)


# ============================================================
# 20. EXTRACT MACROS
# ============================================================

macro_records = []

for food_name, food_data in selected_foods.items():

    fdc_id = food_data["fdc_id"]

    description = food_data["description"]

    rows = food_nutrient[
        food_nutrient["fdc_id"] == fdc_id
    ].copy()

    record = {
        "food_name": food_name,
        "fdc_id": fdc_id,
        "description": description,
        "data_type": "Foundation Food",

        # Explicit USDA reference basis
        "basis_g": 100
    }

    # --------------------------------------------------------
    # Extract each nutrient
    # --------------------------------------------------------

    for output_name, nutrient_id in nutrient_ids.items():

        nutrient_row = rows[
            rows["nutrient_id"] == nutrient_id
        ]

        if nutrient_row.empty:

            record[output_name] = None

        else:

            record[output_name] = (
                nutrient_row.iloc[0][amount_column]
            )

    macro_records.append(record)


# ============================================================
# 21. CREATE FINAL FOOD DATABASE
# ============================================================

food_database = pd.DataFrame(macro_records)


# ============================================================
# 22. CLEAN COLUMN ORDER
# ============================================================

preferred_columns = [
    "food_name",
    "fdc_id",
    "description",
    "data_type",
    "basis_g",
    "protein_g",
    "fat_g",
    "carbohydrate_g"
]

existing_columns = [
    column
    for column in preferred_columns
    if column in food_database.columns
]

food_database = food_database[existing_columns]


# ============================================================
# 23. DISPLAY FINAL DATABASE
# ============================================================

print("\n\n")
print("=" * 80)
print("FINAL FOOD DATABASE")
print("=" * 80)

print(
    food_database.to_string(index=False)
)


# ============================================================
# 24. SAVE DATABASE
# ============================================================

OUTPUT_FILE = DATA_FOLDER / "food_database.csv"

food_database.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n")
print("=" * 80)
print("DONE")
print("=" * 80)

print("Saved:", OUTPUT_FILE)


# ============================================================
# 25. SAVE FOODS THAT NEED MANUAL REVIEW
# ============================================================

if review_foods:

    review_records = []

    for food_name, candidates in review_foods.items():

        for _, row in candidates.iterrows():

            review_records.append({
                "requested_food": food_name,
                "fdc_id": row["fdc_id"],
                "description": row["description"],
                "score": row["score"]
            })

    review_df = pd.DataFrame(review_records)

    REVIEW_FILE = DATA_FOLDER / "food_selection_review.csv"

    review_df.to_csv(
        REVIEW_FILE,
        index=False
    )

    print(
        "Foods requiring review saved to:",
        REVIEW_FILE
    )

else:

    print("No food selections require manual review.")

Loading USDA files...


FileNotFoundError: [Errno 2] No such file or directory: 'food.csv'